# 🧠 Brain Tumor MRI — Model Evaluation on Test Dataset
## Task 3  |  Project 2 — Representation Learning

---

This notebook **loads every saved model** and runs a **fresh evaluation  
on the 1,600 test images** — completely independent of training.

### What this notebook does step by step

| Step | What happens |
|------|-------------|
| 1 | Install libraries & imports |
| 2 | Load config.json from train.ipynb |
| 3 | Rebuild test DataLoader |
| 4 | Load all 4 saved models from disk |
| 5 | Run inference on 1,600 test images per model |
| 6 | Print Accuracy, Precision, Recall, F1 per model |
| 7 | Plot confusion matrix per model |
| 8 | Plot training curves per model |
| 9 | Final ablation comparison table + bar chart |
| 10 | Run TTA on best model |
| 11 | Save all results to JSON |

> **Important:** Run `train.ipynb` and `model_development.ipynb` first.  
> This notebook needs: `config.json` and `saved_models/*.pth`

---
## Step 1 · Install Libraries

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision matplotlib seaborn scikit-learn tqdm -q
print("Libraries ready ✅")

In [ ]:
import os, json, copy, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

---
## Step 2 · Load Configuration from train.ipynb

In [ ]:
with open('config.json') as f:
    cfg = json.load(f)

SEED          = cfg['seed']
IMG_SIZE      = cfg['img_size']
BATCH_SIZE    = cfg['batch_size']
NUM_CLASSES   = cfg['num_classes']
CLASS_NAMES   = cfg['class_names']
IMAGENET_MEAN = cfg['imagenet_mean']
IMAGENET_STD  = cfg['imagenet_std']
SAVE_DIR      = Path('saved_models')

# Fix seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Config loaded ✅")
print(f"  IMG_SIZE   : {IMG_SIZE}")
print(f"  BATCH_SIZE : {BATCH_SIZE}")
print(f"  NUM_CLASSES: {NUM_CLASSES}")
print(f"  Classes    : {CLASS_NAMES}")
print(f"  SEED       : {SEED}")

# Check saved models exist
print(f"\nChecking saved models in '{SAVE_DIR}':")
model_files = {
    'MLP Baseline' : 'MLP_Baseline_best.pth',
    'Custom CNN'   : 'Custom_CNN_best.pth',
    'ResNet-18'    : 'ResNet18_Phase2_best.pth',
    'EfficientNet' : 'EfficientNet_B0_best.pth',
}
for name, fname in model_files.items():
    path = SAVE_DIR / fname
    status = "✅ found" if path.exists() else "❌ NOT FOUND"
    print(f"  {name:<20}: {fname}  {status}")

---
## Step 3 · Rebuild Test DataLoader

In [ ]:
# Dataset path — same as train.ipynb
DATA_ROOT = Path(r"C:\Users\golla\Downloads\Brain Tumor Classification from MRI Images\archive (2)")
TEST_DIR  = DATA_ROOT / 'Testing'

assert TEST_DIR.exists(), f"Test folder not found: {TEST_DIR}"

# Test transforms — NO augmentation, just clean preprocessing
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_dataset = datasets.ImageFolder(str(TEST_DIR), transform=test_transforms)
test_loader  = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,       # NEVER shuffle test data
    num_workers = 0,
    pin_memory  = True,
)

print(f"Test dataset : {len(test_dataset)} images")
print(f"Batches      : {len(test_loader)}")
print(f"Class mapping: {test_dataset.class_to_idx}")

---
## Step 4 · Define Model Architectures

We rebuild each architecture exactly as it was during training,
then load the saved weights from disk.

In [ ]:
# ── Architecture 1: MLP Baseline ─────────────────────────────────────────────
class MLPBaseline(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        flat_dim = 3 * IMG_SIZE * IMG_SIZE   # 150,528
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 2048), nn.BatchNorm1d(2048), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(2048, 512),      nn.BatchNorm1d(512),  nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 128),       nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )
    def forward(self, x): return self.classifier(x)


# ── Architecture 2: Custom CNN ────────────────────────────────────────────────
# IMPORTANT: bias=False because BatchNorm makes conv bias redundant
# self.gap (not self.pool) — must match training exactly
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=dropout),
        )
    def forward(self, x): return self.block(x)

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  dropout=0.10),
            ConvBlock(32,  64,  dropout=0.15),
            ConvBlock(64,  128, dropout=0.20),
            ConvBlock(128, 256, dropout=0.25),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # ← must be 'gap' not 'pool'
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


# ── Architecture 3: ResNet-18 ─────────────────────────────────────────────────
def build_resnet18(num_classes=4):
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(512, num_classes))
    return model


# ── Architecture 4a: EfficientNet-B0 original ────────────────────────────────
def build_efficientnet_b0(num_classes=4):
    model = models.efficientnet_b0(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(model.classifier[1].in_features, num_classes),
    )
    return model


# ── Architecture 4b: EfficientNet-B0 v2 (improved head) ──────────────────────
def build_efficientnet_b0_v2(num_classes=4):
    model = models.efficientnet_b0(weights=None)
    in_f  = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_f, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes),
    )
    return model

print("All architectures defined ✅")
print(f"  MLP      : 150528 → 2048 → 512 → 128 → 4")
print(f"  CNN      : 4 ConvBlocks (bias=False) → GAP → 256 → 128 → 4")
print(f"  ResNet   : pretrained backbone → Dropout → 512 → 4")
print(f"  EffNet   : pretrained backbone → Dropout → 1280 → 4  (or v2: →256→4)")


---
## Step 5 · Load Saved Weights

In [ ]:
def load_model(build_fn, weights_path, name):
    """
    Build a model, load weights, set to eval mode.
    Returns None if weights file not found.
    """
    path = Path(weights_path)
    if not path.exists():
        print(f"  ❌ {name}: weights not found at {path}")
        return None
    model = build_fn(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  ✅ {name:<22}: loaded  ({n_params:>12,} params)  {path.name}")
    return model


print("Loading saved models from disk...\n")

# ── Load MLP ──────────────────────────────────────────────────────────────────
mlp = load_model(
    MLPBaseline,
    SAVE_DIR / 'MLP_Baseline_best.pth',
    'MLP Baseline'
)

# ── Load Custom CNN ───────────────────────────────────────────────────────────
cnn = load_model(
    CustomCNN,
    SAVE_DIR / 'Custom_CNN_best.pth',
    'Custom CNN'
)

# ── Load ResNet-18 ────────────────────────────────────────────────────────────
# Try both possible save names
resnet_path = SAVE_DIR / 'ResNet18_Phase2_best.pth'
if not resnet_path.exists():
    resnet_path = SAVE_DIR / 'ResNet18_best.pth'
resnet = load_model(build_resnet18, resnet_path, 'ResNet-18')

# ── Load EfficientNet-B0 (auto-detect v2 or original) ────────────────────────
eff_path = SAVE_DIR / 'EfficientNet_B0_best.pth'
if not eff_path.exists():
    eff_path = SAVE_DIR / 'EfficientNet_B0_v2_best.pth'

effnet = None
if eff_path.exists():
    # Try v2 first (improved architecture)
    try:
        effnet = load_model(build_efficientnet_b0_v2, eff_path, 'EfficientNet-B0 v2')
    except RuntimeError:
        effnet = load_model(build_efficientnet_b0, eff_path, 'EfficientNet-B0')
else:
    print(f"  ❌ EfficientNet: no weights file found")

print("\nAll available models loaded ✅")

---
## Step 6 · Evaluation Functions

In [ ]:
def run_inference(model, loader):
    """
    Run model on every image in loader.
    Returns y_true, y_pred as numpy arrays.
    No gradients computed — pure inference.
    """
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='  Evaluating', leave=False):
            preds = model(images.to(DEVICE)).argmax(1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_pred)


def print_full_results(y_true, y_pred, model_name):
    """
    Print complete metrics:
      - Overall accuracy
      - Macro precision, recall, F1
      - Per-class breakdown
    Returns metrics dict.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true,    y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true,        y_pred, average='macro', zero_division=0)

    print(f"\n{'═'*58}")
    print(f"  TEST RESULTS: {model_name}")
    print(f"{'═'*58}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  Precision : {prec:.4f}  (macro avg)")
    print(f"  Recall    : {rec:.4f}  (macro avg)")
    print(f"  F1-Score  : {f1:.4f}  (macro avg)")
    print(f"\n{classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)}")

    # Per-class correct / wrong count
    print(f"  Per-class breakdown (out of 400 each):")
    print(f"  {'Class':<15} {'Correct':>8} {'Wrong':>8} {'Recall':>8}")
    print(f"  {'─'*42}")
    for i, cls in enumerate(CLASS_NAMES):
        mask    = (y_true == i)
        correct = ((y_pred == i) & mask).sum()
        wrong   = mask.sum() - correct
        recall  = correct / mask.sum() if mask.sum() > 0 else 0
        print(f"  {cls:<15} {correct:>8} {wrong:>8} {recall:>7.1%}")

    return {'model': model_name, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1,
            'y_true': y_true, 'y_pred': y_pred}


def plot_confusion_matrix(y_true, y_pred, model_name):
    """Annotated confusion matrix heatmap."""
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, linecolor='white', annot_kws={'size': 12})
    ax.set_title(f'{model_name}\nConfusion Matrix (Test Set)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    plt.tight_layout()
    fname = f'eval_cm_{model_name.lower().replace(" ", "_").replace("-","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


print("Evaluation functions ready ✅")

---
## Step 7a · Test Results — MLP Baseline

Weakest model. Expected ~64%. Proves why CNNs are needed.

In [ ]:
if mlp is not None:
    print("Running MLP Baseline on 1,600 test images...")
    y_true_mlp, y_pred_mlp = run_inference(mlp, test_loader)
    results_mlp = print_full_results(y_true_mlp, y_pred_mlp, 'MLP Baseline')
    plot_confusion_matrix(y_true_mlp, y_pred_mlp, 'MLP Baseline')
else:
    print("MLP not loaded — skipping")

---
## Step 7b · Test Results — Custom CNN

Built from scratch. Expected ~85%. Shows value of spatial features.

In [ ]:
if cnn is not None:
    print("Running Custom CNN on 1,600 test images...")
    y_true_cnn, y_pred_cnn = run_inference(cnn, test_loader)
    results_cnn = print_full_results(y_true_cnn, y_pred_cnn, 'Custom CNN')
    plot_confusion_matrix(y_true_cnn, y_pred_cnn, 'Custom CNN')
else:
    print("Custom CNN not loaded — skipping")

---
## Step 7c · Test Results — ResNet-18

Transfer learning. Expected ~95%. Shows value of pretrained weights.

In [ ]:
if resnet is not None:
    print("Running ResNet-18 on 1,600 test images...")
    y_true_rn, y_pred_rn = run_inference(resnet, test_loader)
    results_resnet = print_full_results(y_true_rn, y_pred_rn, 'ResNet-18')
    plot_confusion_matrix(y_true_rn, y_pred_rn, 'ResNet-18')
else:
    print("ResNet-18 not loaded — skipping")

---
## Step 7d · Test Results — EfficientNet-B0

Best model. Expected ~96-97%. Compound scaling + improved head.

In [ ]:
if effnet is not None:
    print("Running EfficientNet-B0 on 1,600 test images...")
    y_true_en, y_pred_en = run_inference(effnet, test_loader)
    results_effnet = print_full_results(y_true_en, y_pred_en, 'EfficientNet-B0')
    plot_confusion_matrix(y_true_en, y_pred_en, 'EfficientNet-B0')
else:
    print("EfficientNet not loaded — skipping")

---
## Step 8 · Final Ablation Comparison Table

All 4 models side by side on the same 1,600 test images.

In [ ]:
# Collect all available results
all_results = []
for name, res in [('MLP Baseline', 'results_mlp'),
                   ('Custom CNN',   'results_cnn'),
                   ('ResNet-18',    'results_resnet'),
                   ('EfficientNet-B0', 'results_effnet')]:
    if res in dir():
        all_results.append(eval(res))

if not all_results:
    print("No results found — run cells above first")
else:
    print(f"\n{'═'*72}")
    print(f"  ABLATION TABLE — Test Set Results (1,600 images)")
    print(f"{'═'*72}")
    print(f"  {'Model':<22} {'Accuracy':>10} {'Precision':>11} {'Recall':>9} {'F1':>10}")
    print(f"{'─'*72}")
    for r in all_results:
        marker = " ← best" if r['accuracy'] == max(x['accuracy'] for x in all_results) else ""
        print(f"  {r['model']:<22} {r['accuracy']*100:>9.2f}%"
              f"  {r['precision']:>10.4f}  {r['recall']:>8.4f}  {r['f1']:>9.4f}{marker}")
    print(f"{'═'*72}")

    # Improvement over MLP
    if len(all_results) > 1:
        mlp_acc = all_results[0]['accuracy']
        best_acc = max(r['accuracy'] for r in all_results)
        print(f"\n  Improvement (MLP → best model): +{(best_acc-mlp_acc)*100:.2f} percentage points")

---
## Step 9 · Visual Comparison Bar Chart

In [ ]:
if all_results:
    metrics     = ['accuracy', 'precision', 'recall', 'f1']
    labels      = ['Accuracy (%)', 'Precision (macro)', 'Recall (macro)', 'F1-Score (macro)']
    colors      = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
    model_names = [r['model'] for r in all_results]
    x           = np.arange(len(model_names))
    width       = 0.18

    fig, axes = plt.subplots(1, 4, figsize=(18, 6))
    fig.suptitle('Model Comparison — Test Set (1,600 images)',
                 fontsize=14, fontweight='bold', y=1.02)

    for ax, key, label, color in zip(axes, metrics, labels, colors):
        vals = [r[key] * (100 if key == 'accuracy' else 1) for r in all_results]
        bars = ax.bar(model_names, vals, color=color, alpha=0.85, edgecolor='white', linewidth=0.8)

        # Value labels on top of bars
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.003,
                    f'{val:.2f}{"%" if key=="accuracy" else ""}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_ylim(0, max(vals) * 1.12)
        ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('eval_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: eval_model_comparison.png")

---
## Step 10 · Test-Time Augmentation (TTA) on Best Model

TTA runs each test image through the model **10 times** with random augmentations,
then averages all predictions. More stable than a single prediction.

> **Note:** Takes ~5 minutes — each of 1,600 images predicted 10 times.

In [ ]:
# Pick best available model for TTA
best_model = None
best_name  = ''
best_acc   = 0.0
for name, model, res_name in [
        ('EfficientNet-B0', 'effnet',  'results_effnet'),
        ('ResNet-18',       'resnet',  'results_resnet'),
        ('Custom CNN',      'cnn',     'results_cnn')]:
    if name in dir() or model in dir():
        m = eval(model) if model in dir() else None
        r = eval(res_name) if res_name in dir() else None
        if m is not None and r is not None and r['accuracy'] > best_acc:
            best_model = m
            best_name  = name
            best_acc   = r['accuracy']

if best_model is None:
    print("No model available for TTA")
else:
    print(f"Running TTA on: {best_name}  (baseline accuracy: {best_acc*100:.2f}%)")

    _tta_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    ])

    def predict_tta(model, img_tensor, n_aug=10):
        model.eval()
        probs = torch.zeros(1, NUM_CLASSES).to(DEVICE)
        with torch.no_grad():
            probs += torch.softmax(model(img_tensor.to(DEVICE)), dim=1)
            for _ in range(n_aug - 1):
                aug = _tta_tf(img_tensor)
                probs += torch.softmax(model(aug.to(DEVICE)), dim=1)
        return (probs / n_aug).argmax(1).item()

    y_true_tta, y_pred_tta = [], []
    for images, labels in tqdm(test_loader, desc=f'TTA (n=10)'):
        for i in range(images.size(0)):
            y_pred_tta.append(predict_tta(best_model, images[i:i+1]))
        y_true_tta.extend(labels.numpy())

    y_true_tta = np.array(y_true_tta)
    y_pred_tta = np.array(y_pred_tta)

    results_tta = print_full_results(y_true_tta, y_pred_tta, f'{best_name} + TTA')
    delta = (results_tta['accuracy'] - best_acc) * 100
    print(f"\n  Without TTA : {best_acc*100:.2f}%")
    print(f"  With    TTA : {results_tta['accuracy']*100:.2f}%")
    print(f"  Improvement : {delta:+.2f} percentage points")

---
## Step 11 · Save All Results

In [ ]:
import json as _json

# Collect metrics (exclude y_true/y_pred arrays for JSON serialisation)
def to_json_safe(r):
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v)
            for k, v in r.items() if k not in ('y_true', 'y_pred')}

save_results = []
for res_name in ['results_mlp', 'results_cnn', 'results_resnet',
                 'results_effnet', 'results_tta']:
    if res_name in dir():
        save_results.append(to_json_safe(eval(res_name)))

with open('eval_results.json', 'w') as f:
    _json.dump(save_results, f, indent=2)

print("Saved: eval_results.json")
print("\nAll output files:")
for fname in sorted(Path('.').glob('eval_*.png')) :
    print(f"  {fname}")
print("  eval_results.json")
print("\n✅ Evaluation complete!")